# Machine Learning Project
# Kylle Waldie

# Pokemon Grading Tool

## Ebay API

### Imports

In [12]:
import requests
import base64
import csv
import re
import os
from dotenv import load_dotenv
import cv2
import numpy as np

ModuleNotFoundError: No module named 'cv2'

### Config

In [2]:
load_dotenv()  # loads .env into environment variables

CLIENT_ID = os.getenv("EBAY_CLIENT_ID")
CLIENT_SECRET = os.getenv("EBAY_CLIENT_SECRET")
MARKETPLACE_ID = "EBAY_US"

print("Client ID loaded:", CLIENT_ID is not None)
print("Client Secret loaded:", CLIENT_SECRET is not None)


Client ID loaded: True
Client Secret loaded: True


### Authorization

In [3]:
def get_ebay_access_token(client_id, client_secret):
    credentials = f"{client_id}:{client_secret}"
    encoded = base64.b64encode(credentials.encode()).decode()

    url = "https://api.ebay.com/identity/v1/oauth2/token"

    headers = {
        "Authorization": f"Basic {encoded}",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    data = {
        "grant_type": "client_credentials",
        "scope": "https://api.ebay.com/oauth/api_scope"
    }

    response = requests.post(url, headers=headers, data=data)
    response.raise_for_status()

    return response.json()["access_token"]

### Grade Extraction

In [4]:
def extract_psa_grade(title):
    match = re.search(r'PSA\s?-?\s?(\d{1,2})', title, re.IGNORECASE)
    return match.group(1) if match else None

### Download Images

In [5]:
def download_image(url, filepath):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        with open(filepath, "wb") as f:
            f.write(response.content)

        return True
    except Exception as e:
        print(f"Failed to download {url}: {e}")
        return False

### Download By PSA

In [6]:
def download_psa_images(psa_data, base_dir="dataset"):
    os.makedirs(base_dir, exist_ok=True)

    counters = {}

    for grade, image_url in psa_data:
        grade_dir = os.path.join(base_dir, f"PSA_{grade}")
        os.makedirs(grade_dir, exist_ok=True)

        counters.setdefault(grade, 0)
        counters[grade] += 1

        filename = f"psa_{grade}_{counters[grade]}.jpg"
        filepath = os.path.join(grade_dir, filename)

        download_image(image_url, filepath)

    return counters

### eBay Search

In [7]:
def search_psa_items(access_token, query="Pokemon PSA graded", limit=100, offset=0, marketplace_id="EBAY_US"):
    import requests

    url = "https://api.ebay.com/buy/browse/v1/item_summary/search"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "X-EBAY-C-MARKETPLACE-ID": marketplace_id
    }

    params = {
        "q": query,
        "limit": limit,
        "offset": offset
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json().get("itemSummaries", [])


### Extract Grade & Images

In [8]:
def extract_grade_and_images(items):
    results = []

    for item in items:
        title = item.get("title", "")
        grade = extract_psa_grade(title)

        if not grade:
            continue

        image_url = item.get("image", {}).get("imageUrl")

        if image_url:
            results.append((grade, image_url))

    return results

### Higher Res Images

In [9]:
def get_highres_image_url(url):
    # Replace small-size suffix with s-l1600 for best standard resolution
    return re.sub(r's-l\d+\.', 's-l1600.', url)

### Saving CSV File

In [10]:
def save_to_csv(data, filename="psa_grades_images.csv"):
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["PSA_Grade", "Image_URL"])
        writer.writerows(data)

### Running Funcs

In [11]:
access_token = get_ebay_access_token(CLIENT_ID, CLIENT_SECRET)

all_items = []
total_to_fetch = 250  # total items you want
limit = 100            # max per request allowed by eBay

for offset in range(0, total_to_fetch, limit):
    items = search_psa_items(
        access_token,
        query="Pokemon PSA graded",
        limit=limit,
        offset=offset,      # <-- offset ensures pagination
        marketplace_id=MARKETPLACE_ID
    )
    all_items.extend(items)

psa_data = extract_grade_and_images(all_items)

save_to_csv(psa_data)

counts = download_psa_images(psa_data)
counts

{'9': 80, '10': 87, '8': 25, '7': 10, '1': 1, '4': 2, '6': 6, '5': 3}